# Control Trainer

> Fill in a module description here

In [ ]:
#| default_exp trainers.world_model

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math
import torch
import os
import torch.nn as nn
import torchvision.utils as vutils
import wandb
import hydra
from pathlib import Path
from fastcore.utils import patch
from loguru import logger
from omegaconf import DictConfig
from einops import rearrange
import torch.nn.functional as F
from c3jepa_wm.utils.checkpointer import RetrospectiveCheckpointer
from c3jepa_wm.utils import channel


In [ ]:
#| export
class TrainerScheduler:
    def __init__(self, wm_optimizer, power_optimizer):
        self.wm_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            wm_optimizer, mode='min', factor=0.5, patience=5
        )
        self.power_scheduler = self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                power_optimizer, mode='min', factor=0.5, patience=5
        )

    def step(self, val_loss):
        jepa_val_loss = val_loss[0]
        self.wm_scheduler.step(jepa_val_loss)

        power_val_loss = val_loss[1]
        self.power_scheduler.step(power_val_loss)
        

In [ ]:
#| export
class BaseTrainer:
    def __init__(self, 
                 data_module, 
                 device, 
                 slurm_jobid= None, 
                 wm_lr=1e-4, 
                 power_lr=3e-4,
                 epochs=100, 
                 project_name="c3jepa_wm",
                 ckp_dir= "checkpoints",
                 save_dir= "outputs"):
        
        self.data_module = data_module
        
        self.device = device
        
        self.data_module.setup()
        self.train_loader = self.data_module.train_dataloader()
        self.val_loader = self.data_module.val_dataloader()

        self.slurm_jobid = slurm_jobid if slurm_jobid else "default_job"
        self.wm_lr = wm_lr
        self.power_lr = power_lr
        self.epochs = epochs
        self.project_name = project_name
        self.ckp_dir = ckp_dir
        self.save_dir = save_dir

        

    def init_optimizer(self, model, lr, weight_decay=0.01):
        return torch.optim.AdamW(
                list(model.parameters()),
                lr=lr,
                weight_decay=weight_decay
            )
    
    def train_epoch(self, epoch):
        raise NotImplementedError("train_epoch method must be implemented by subclasses.")   

    def validate_epoch(self, epoch):
        raise NotImplementedError("validate method must be implemented by subclasses.")
    

In [ ]:
#| export
class WMTrainer(BaseTrainer):
    def __init__(self, data_module, model, device, wm_lr, power_lr, history_size, num_preds, lambda_sigreg, lambda_pow, lambda_value, lambda_quality, lambda_send, **kwargs):
        super().__init__(
            data_module= data_module,
            device= device, 
            wm_lr= wm_lr,
            power_lr= power_lr,
            **kwargs)
        
        self.history_size = history_size
        self.num_preds = num_preds

        self.lambda_sigreg = lambda_sigreg
        self.lambda_pow = lambda_pow
        self.lambda_value = lambda_value
        self.lambda_quality = lambda_quality
        self.lambda_send = lambda_send

        self.vqvae = model["vqvae"].to(device)
        self.power_net = model["power_net"].to(device)
        self.model = model["jepa"].to(device)


        self.model.predictor.msg_token_embedding.weight.data[:self.vqvae.vq_layer.K].copy_(
            self.vqvae.vq_layer.embedding.detach()
        )
        self.power_net.msg_embedding.weight.data.copy_(
            self.vqvae.vq_layer.embedding.detach()
        )

        self.wm_optimizer = self.init_optimizer(self.model, lr=self.wm_lr, weight_decay=1e-3)
        self.power_optimizer = self.init_optimizer(self.power_net, lr=self.power_lr, weight_decay=1e-4)
        self.scheduler = TrainerScheduler(self.wm_optimizer, self.power_optimizer)
        
        
        self.save_dir = Path(hydra.utils.to_absolute_path(self.save_dir))
        self.save_dir.mkdir(exist_ok=True, parents=True)

        self.ck_pointer = RetrospectiveCheckpointer(project_name= self.project_name,
                                                    ckp_dir= self.ckp_dir,
                                                    slurm_jobid= self.slurm_jobid,
                                                    agent_id= "vqvae_trainer",
                                                    rank= 0, 
                                                    n_best=3)

        # Create output directories for visual inspection
        os.makedirs(os.path.join(self.save_dir, "Reconstructions"), exist_ok=True)


    @torch.no_grad()
    def get_msg_indices(self, sender_pov_seq):
        """Helper function to get message indices from the pretrained VQ-VAE for a given sender POV sequence.
        Args:
            sender_pov_seq: Tensor of shape (B, T, C, H, W) representing the sender's point-of-view image sequence.
        Returns:
            msg_indices: Tensor of shape (B*T, 49) 
        """
        B, T, C, H, W = sender_pov_seq.shape
        sender_pov_flat = rearrange(sender_pov_seq.to(self.device), "b t c h w -> (b t) c h w")  # (B*T, C, H, W)
        msg_indices_flat = self.vqvae.get_message_indices(sender_pov_flat)  # (B*T, 7, 7)
        msg_indices = rearrange(msg_indices_flat, "b h w -> b (h w)", b=B*T)   # (B*T, 49)
        return msg_indices


    def train_epoch(self, epoch):
        self.model.train()
        self.power_net.train()
        total_loss_jepa = 0.0
        total_loss_power = 0.0
        for batch_idx, batch in enumerate(self.train_loader):
            batch = {k: v.to(self.device) for k, v in batch.items()}
            # 0. Zero grads
            self.wm_optimizer.zero_grad()
            self.power_optimizer.zero_grad()

            # 1. Get message indices from pretrained VQ-VAE
            msg_indices = self.get_msg_indices(
                batch["sender_pov"]
            )  # (B*T, 49)
            
            B = batch["sender_pov"].shape[0]
            T = self.history_size
            
            csi_flat = rearrange(
                batch["sender_csi"].to(self.device), "b t n d -> (b t) n d"
            )  # (B*T, N, D)

            # 2. PowerNet: schedule and power decisions
            logger.info(f"PowerNet input msg_indices shape: {msg_indices.shape}, csi_flat shape: {csi_flat.shape}")
            schedule, power = self.power_net(msg_indices, csi_flat)  # (B*T, N, 1)

            # 3. Channel
            logger.info(f"Channel input schedule shape: {schedule.shape}, power shape: {power.shape}, msg_indices shape: {msg_indices.shape}, csi_flat shape: {csi_flat.shape}")
            received_msg = channel(
                schedule, power, msg_indices, csi_flat, device=self.device
            )  # (B*T, 49)

            received_msg = rearrange(
                received_msg, "(b t) msg_dim -> b t msg_dim", b=B, t=T
            )  # (B, T, 49)

            # 4. Encode receiver observations
            logger.info(f"Encoding receiver observations with shape {batch['pixels'].shape}")
            output = self.model.encode(batch)
            emb = output["emb"]                          # (B, T+1, D)
            act_emb = output["act_emb"].reshape(B, T, -1)  # (B, T, act_emb_dim)

            # 5. Slice context and target
            ctx_emb = emb[:, :T]                         # (B, T, D)
            ctx_act = act_emb[:, :T]                     # (B, T, act_emb_dim)
            ctx_msg = received_msg[:, :T]                # (B, T, 49) — raw indices, embedded inside predictor
            tgt_emb = emb[:, self.num_preds:]            # (B, T, D)

            # 6. Predict
            logger.info(f"Predicting from world model with context emb shape {ctx_emb.shape}, action emb shape {ctx_act.shape}, and msg shape {ctx_msg.shape}")
            pred_emb = self.model.predict(ctx_emb, ctx_act, ctx_msg)  # (B, T, D)

            # 7. JEPA loss
            output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, self.lambda_sigreg)

            # 8. TX loss (uses pred_loss so compute before backward)
            power_loss = self.power_net.loss_fn(
                self.model, T, ctx_emb, ctx_act,
                msg_indices, tgt_emb, schedule, power,
                output["pred_loss"], self.lambda_value, self.lambda_pow, self.lambda_send
            )
            for k, v in power_loss.items():
                output[k] = v

            with torch.no_grad():
                logger.info(f"Computing perfect communication loss for quality metric.")
                msg_indices = rearrange(msg_indices, "(b t) msg_dim -> b t msg_dim", msg_dim=49, b=B, t=T)   # (B, T, 49)
                ctx_msg = msg_indices[:, :T] # (B, ctx_len, msg_dim)
                pred_emb_perfect_msg = self.model.predict(ctx_emb, ctx_act, ctx_msg) # pred with perfect comm.
                perfect_loss = (pred_emb_perfect_msg - tgt_emb).pow(2).mean()
                quality = perfect_loss - output["pred_loss"]
        
            loss_dict = {k: v for k, v in output.items() if "loss" in k}
            loss_dict["quality"] = quality
            wandb.log({f"train_{k}": v.item() for k, v in loss_dict.items()})

            # 9. Backward in correct order
            output['jepa_loss'].backward(retain_graph=True)
            output['tx_loss'].backward()

            total_loss_jepa += output['jepa_loss'].item()
            total_loss_power += output['tx_loss'].item()

            # 10. Clip and step
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(self.power_net.parameters(), max_norm=1.0)
            self.wm_optimizer.step()
            self.power_optimizer.step()
            logger.info(f"Completed batch {batch_idx+1}/{len(self.train_loader)} for epoch {epoch} with JEPA loss {output['jepa_loss'].item():.4f} and Power loss {output['tx_loss'].item():.4f}")

        avg_loss_jepa = total_loss_jepa / len(self.train_loader)
        avg_loss_power = total_loss_power / len(self.train_loader)
        print(f"Epoch [{epoch}/{self.epochs}] - Train JEPA Loss: {avg_loss_jepa:.4f}, Power Loss: {avg_loss_power:.4f}")
        wandb.log({"train_avg_jepa_loss": avg_loss_jepa, "train_avg_power_loss": avg_loss_power}, step=epoch)
        return avg_loss_jepa, avg_loss_power
    
    @torch.no_grad()
    def validate_epoch(self, epoch):
        self.model.eval()
        self.power_net.eval()
        total_loss_jepa = 0.0
        total_loss_power = 0.0
        for batch_idx, batch in enumerate(self.val_loader):
            batch = {k: v.to(self.device) for k, v in batch.items()}
            # 1. Get message indices from pretrained VQ-VAE
            msg_indices = self.get_msg_indices(
                batch["sender_pov"]
            )  # (B*T, 49) 
            B = batch["sender_pov"].shape[0]
            T = self.history_size
            csi_flat = rearrange(
                batch["sender_csi"].to(self.device), "b t n d -> (b t) n d"
            )  # (B*T, N, D)
            # 2. PowerNet: schedule and power decisions
            schedule, power = self.power_net(msg_indices, csi_flat)  # (B*T, N, 1)
            # 3. Channel
            received_msg = channel(
                schedule, power, msg_indices, csi_flat, device=self.device
            )  # (B*T, 49)
            received_msg = rearrange(
                received_msg, "(b t) msg_dim -> b t msg_dim", b=B, t=T
            )  # (B, T, 49)
            # 4. Encode receiver observations
            output = self.model.encode(batch)
            emb = output["emb"]                          # (B, T+1, D)
            act_emb = output["act_emb"].reshape(B, T, -1)  # (B, T, act_emb_dim)
            # 5. Slice context and target
            ctx_emb = emb[:, :T]                         # (B, T, D)
            ctx_act = act_emb[:, :T]                     # (B, T, act_emb_dim)
            ctx_msg = received_msg[:, :T]                # (B, T, 49) — raw indices, embedded inside predictor
            tgt_emb = emb[:, self.num_preds:]            # (B, T, D)
            # 6. Predict
            pred_emb = self.model.predict(ctx_emb, ctx_act, ctx_msg)  # (B, T, D)
            # 7. JEPA and powenet losses
            output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, self.lambda_sigreg)
            power_loss = self.power_net.loss_fn(
                self.model, T, ctx_emb, ctx_act,
                msg_indices, tgt_emb, schedule, power,
                output["pred_loss"], self.lambda_value, self.lambda_pow, self.lambda_send
            )
            for k, v in power_loss.items():
                output[k] = v
            total_loss_jepa += output['jepa_loss'].item()    
            total_loss_power += output['tx_loss'].item()

        avg_loss_jepa = total_loss_jepa / len(self.val_loader)
        avg_loss_power = total_loss_power / len(self.val_loader)
        print(f"Epoch [{epoch}/{self.epochs}] - Val JEPA Loss: {avg_loss_jepa:.4f}, Val Power Loss: {avg_loss_power:.4f}")
        wandb.log({"val_avg_jepa_loss": avg_loss_jepa, "val_avg_power_loss": avg_loss_power}, step=epoch)

        return avg_loss_jepa, avg_loss_power


In [ ]:
#| export
@patch
def checkpoint(self: WMTrainer, epoch, val_loss):
    checkpoint_state = {
        "epoch": epoch,
        "wm_model_state_dict": self.model.state_dict(),
        "wm_optimizer_state_dict": self.wm_optimizer.state_dict(),
        "power_model_state_dict": self.power_net.state_dict(),
        "power_optimizer_state_dict": self.power_optimizer.state_dict(),
        "wm_val_loss": val_loss[0],
        "power_val_loss": val_loss[1],
    }
    self.ck_pointer.save_checkpoint(state= checkpoint_state, current_acc= -val_loss[0], step= epoch)
    

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()